# Example 03 · Agent Harness Patterns

**Source:** [LangChain Agents — The Harness](https://docs.langchain.com/oss/python/langchain/agents)

## Core Idea: Agent = Model + Harness

The **model** reasons. The **harness** is everything surrounding it — system prompt, tools,
structured output schema, per-run context, checkpointer, and middleware.

This notebook demonstrates three harness features that go beyond a plain ReAct loop:

| Demo | Feature | Key API |
|------|---------|---------|
| A | **Streaming** — watch tool calls arrive in real time | `agent.stream(..., stream_mode="values")` |
| B | **Structured output** — agent returns a validated Pydantic object | `response_format=BookReport` |
| C | **Context schema** — per-run user data injected into tools | `context_schema=ReaderContext` + `ToolRuntime` |

In [1]:
import sys
from dataclasses import dataclass
from pathlib import Path

_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

from common.env import get_env  # noqa: F401
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# ── Mock book database ────────────────────────────────────────────────────────
_BOOKS = {
    "the great gatsby": {
        "author": "F. Scott Fitzgerald (1925)", "genre": "Literary fiction",
        "themes": ["The American Dream", "class and wealth", "obsession"],
        "plot": "Nick Carraway befriends millionaire Jay Gatsby, who throws lavish parties hoping to win back his lost love Daisy.",
    },
    "1984": {
        "author": "George Orwell (1949)", "genre": "Dystopian fiction",
        "themes": ["totalitarianism", "surveillance", "propaganda"],
        "plot": "Winston Smith secretly rebels against the all-seeing Party in a totalitarian society ruled by Big Brother.",
    },
    "dune": {
        "author": "Frank Herbert (1965)", "genre": "Science fiction",
        "themes": ["ecology", "religion", "politics", "power"],
        "plot": "Paul Atreides is sent to govern the desert planet Arrakis and is ultimately betrayed, beginning a journey toward destiny.",
    },
}

# ── Context schema ─────────────────────────────────────────────────────────────
@dataclass
class ReaderContext:
    """Per-run user data. Passed via context= in agent.invoke(); reaches tools via ToolRuntime."""
    reader_name: str
    preferred_style: str  # "bullet points" | "casual" | "academic"

# ── Tool that accesses context via ToolRuntime ────────────────────────────────
@tool
def lookup_book(title: str, runtime: ToolRuntime[ReaderContext]) -> str:
    """Look up a book and return content formatted for the current reader's preferred style."""
    # runtime.context is the ReaderContext passed via context= — it arrives here
    # through the harness without appearing in the LLM's message history.
    reader = runtime.context
    if reader:
        print(f"  [tool] context → name={reader.reader_name!r}, style={reader.preferred_style!r}")

    key = title.lower().strip()
    book = _BOOKS.get(key)
    if not book:
        return f"Not found. Available: {', '.join(t.title() for t in _BOOKS)}"

    # Format output based on reader style so the LLM produces visibly different responses
    if reader and reader.preferred_style == "bullet points":
        return (
            f"For {reader.reader_name} (bullet-point format):\n"
            f"• Title: {title.title()}\n"
            f"• Author: {book['author']}\n"
            f"• Genre: {book['genre']}\n"
            f"• Themes: {', '.join(book['themes'])}\n"
            f"• Plot: {book['plot']}"
        )
    name  = reader.reader_name if reader else "reader"
    style = f" ({reader.preferred_style} style)" if reader else ""
    return (
        f"For {name}{style}:\n"
        f"Title: {title.title()} by {book['author']}\n"
        f"Genre: {book['genre']}\n"
        f"Themes: {', '.join(book['themes'])}\n"
        f"Plot: {book['plot']}"
    )

handler = create_langfuse_handler()

def new_config(trace_name: str, thread_id: str = None) -> dict:
    cfg = build_langfuse_config(handler, session_id="s03-nb", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id or str(uuid7())}
    return cfg

print("✓ Setup, book DB, and tools ready")

✓ Setup, book DB, and tools ready


## Demo A · Streaming

`agent.stream()` with `stream_mode="values"` yields the **full agent state** after every
node execution. Inspect `chunk["messages"][-1]` at each step to see tool calls and replies
as they arrive — no waiting for the final answer.

In [2]:
streaming_agent = create_agent(
    model=create_llm(),
    tools=[lookup_book],
    system_prompt="You are a knowledgeable book assistant. Use lookup_book to answer questions.",
    checkpointer=InMemorySaver(),
)

question_a = "Tell me about '1984' by George Orwell."
print(f"Q: {question_a}\n")

for chunk in streaming_agent.stream(
    {"messages": [{"role": "user", "content": question_a}]},
    config=new_config("Demo A — Streaming"),
    stream_mode="values",
):
    latest = chunk["messages"][-1]
    if isinstance(latest, HumanMessage):
        pass  # skip echoing the user's own message
    elif isinstance(latest, AIMessage) and latest.tool_calls:
        names = [tc["name"] for tc in latest.tool_calls]
        print(f"  [tool call] → {', '.join(names)}")
    elif isinstance(latest, AIMessage) and latest.content:
        print(f"  [reply]\n{latest.content}")

Q: Tell me about '1984' by George Orwell.

  [tool call] → lookup_book
  [reply]
"1984" by George Orwell, published in 1949, is a classic work of dystopian fiction. The novel explores themes of totalitarianism, surveillance, and propaganda. 

The plot follows Winston Smith, who secretly rebels against the oppressive regime of the Party, which is led by the omnipresent figure known as Big Brother. In this society, individual freedom is suppressed, and the government constantly monitors its citizens, manipulating truth and reality to maintain control.


## Demo B · Structured Output (`response_format`)

Pass a Pydantic model as `response_format` to receive a **validated object** instead of free text.
The agent fills every field by combining tool results with its own reasoning.
The result lives under `result["structured_response"]` — guaranteed schema, no text parsing needed.

In [3]:
class BookReport(BaseModel):
    """Structured book report — every field populated by the agent."""
    title: str           = Field(description="Book title")
    author: str          = Field(description="Author name and publication year")
    one_line_summary: str = Field(description="One sentence capturing the book's essence")
    themes: list[str]   = Field(description="Up to 3 main themes")
    recommended_for: str = Field(description="Who would enjoy this book most")
    rating: float       = Field(description="Your rating from 1.0 to 5.0", ge=1.0, le=5.0)

structured_agent = create_agent(
    model=create_llm(),
    tools=[lookup_book],
    system_prompt=(
        "You are a literary critic. "
        "Use lookup_book to research the book, then fill every field of the report accurately."
    ),
    response_format=BookReport,
    checkpointer=InMemorySaver(),
)

question_b = "Write a structured report on 'Dune'."
print(f"Q: {question_b}\n")

result_b = structured_agent.invoke(
    {"messages": [{"role": "user", "content": question_b}]},
    config=new_config("Demo B — Structured Output"),
)

report: BookReport = result_b["structured_response"]
print(f"Title           : {report.title}")
print(f"Author          : {report.author}")
print(f"One-line summary: {report.one_line_summary}")
print(f"Themes          : {', '.join(report.themes)}")
print(f"Recommended for : {report.recommended_for}")
print(f"Rating          : {report.rating} / 5.0")

Q: Write a structured report on 'Dune'.

Title           : Dune
Author          : Frank Herbert (1965)
One-line summary: A sweeping epic of politics, religion, and ecology set on the desert planet of Arrakis, where young Paul Atreides must navigate betrayal and destiny.
Themes          : ecology, religion, politics
Recommended for : Fans of science fiction and epic narratives, particularly those interested in complex world-building and philosophical themes.
Rating          : 5.0 / 5.0


## Demo C · Context Schema (`context_schema` + `ToolRuntime`)

`context_schema` declares the shape of per-run user data. The data is passed via `context=`
at invocation time and reaches tool functions through `ToolRuntime` — injected by the harness.

**Why this is different from just adding context to the system prompt:**
- The LLM never sees `reader_name` or `preferred_style` as message text
- Context flows directly into the tool function via dependency injection
- Sensitive values (user IDs, API keys, feature flags) stay out of the LLM conversation

In this demo the tool formats its output differently for each reader, so both the tool output
and the final LLM response are visibly different — proving the context arrived in the tool.

In [4]:
context_agent = create_agent(
    model=create_llm(),
    tools=[lookup_book],
    system_prompt=(
        "You are a personal book assistant. "
        "Use lookup_book to fetch book info, then summarize following the format in the tool output."
    ),
    context_schema=ReaderContext,
    checkpointer=InMemorySaver(),
)

question_c = "Summarize 'The Great Gatsby' for me."

readers = [
    ReaderContext(reader_name="Alice", preferred_style="bullet points"),
    ReaderContext(reader_name="Bob",   preferred_style="casual and conversational"),
]

for reader in readers:
    print(f"\n{'─'*50}")
    print(f"Reader: {reader.reader_name}  |  Style: {reader.preferred_style}")
    print("─" * 50)
    result_c = context_agent.invoke(
        {"messages": [{"role": "user", "content": question_c}]},
        config=new_config(f"Demo C — Context ({reader.reader_name})"),
        context=reader,
    )
    print(result_c["messages"][-1].content)

print(f"\nTraces: {get_langfuse_host()}")


──────────────────────────────────────────────────
Reader: Alice  |  Style: bullet points
──────────────────────────────────────────────────
  [tool] context → name='Alice', style='bullet points'
Here’s a summary of "The Great Gatsby":

- **Title**: The Great Gatsby
- **Author**: F. Scott Fitzgerald (1925)
- **Genre**: Literary fiction
- **Themes**: The American Dream, class and wealth, obsession
- **Plot**: Nick Carraway befriends millionaire Jay Gatsby, who throws lavish parties hoping to win back his lost love, Daisy.

──────────────────────────────────────────────────
Reader: Bob  |  Style: casual and conversational
──────────────────────────────────────────────────
  [tool] context → name='Bob', style='casual and conversational'
**Title:** The Great Gatsby  
**Author:** F. Scott Fitzgerald (1925)  
**Genre:** Literary fiction  
**Themes:** The American Dream, class and wealth, obsession  

**Plot Summary:** The story follows Nick Carraway, who becomes friends with the mysterious 